<a href="https://colab.research.google.com/github/liwiaflorkiwicz/EEG_dimensiality_reduction/blob/preprocessing/OPSI_preprocessing1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Obliczeniowe podstawy sztucznej inteligencji - projekt

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("houssamboukhecham/eeg-motor-movementimagery-hierarchical")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'eeg-motor-movementimagery-hierarchical' dataset.
Path to dataset files: /kaggle/input/eeg-motor-movementimagery-hierarchical


In [3]:
import os
import glob
import pandas as pd

print("Zawartość folderu:")
print(os.listdir(path))

Zawartość folderu:
['eeg-mmi-kaggle']


In [4]:
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)

print(f"Found {len(csv_files)} CSV files.")

# sample file
sample_file = csv_files[0]
print(f"\nReading file: {sample_file}\n")

# read as pandas df
df = pd.read_csv(sample_file)

display(df.head())

Found 6583 CSV files.

reading file: /kaggle/input/eeg-motor-movementimagery-hierarchical/eeg-mmi-kaggle/S035_Task1_6.csv



,Fc5.,Fc3.,Fc1.,Fcz.,Fc2.,Fc4.,Fc6.,C5..,C3..,C1..,...,P8..,Po7.,Po3.,Poz.,Po4.,Po8.,O1..,Oz..,O2..,Iz..
0,108.0,110.0,119.0,116.0,114.0,107.0,104.0,100.0,103.0,110.0,...,-15.0,-20.0,-23.0,-31.0,-39.0,-54.0,-41.0,-45.0,-50.0,-54.0
1,-65.0,-67.0,-70.0,-67.0,-70.0,-70.0,-70.0,-90.0,-82.0,-91.0,...,-40.0,-36.0,-32.0,-29.0,-32.0,-42.0,-23.0,-27.0,-24.0,-35.0
2,-30.0,-24.0,-20.0,-12.0,-13.0,-1.0,6.0,7.0,14.0,13.0,...,115.0,114.0,115.0,109.0,114.0,97.0,107.0,98.0,100.0,81.0
3,72.0,72.0,73.0,81.0,78.0,79.0,76.0,71.0,72.0,58.0,...,-67.0,-71.0,-73.0,-87.0,-83.0,-85.0,-82.0,-88.0,-87.0,-92.0
4,-108.0,-106.0,-109.0,-96.0,-98.0,-96.0,-91.0,-102.0,-92.0,-106.0,...,34.0,20.0,26.0,30.0,48.0,41.0,43.0,37.0,52.0,37.0


In [9]:
# !pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 59.1 MB/s eta 0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [7]:
import mne

# sampling frequency
sfreq = 160

# channels names
ch_names = list(df.columns)

info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')

# data in MNE format (n_channels, n_samples).
data = df.values.T

# raw eeg instance
raw = mne.io.RawArray(data, info)

Creating RawArray with float64 data, n_channels=64, n_times=640
    Range : 0 ... 639 =      0.000 ...     3.994 secs
Ready.


In [16]:
import glob
from collections import Counter

# Listy do przechowywania wyciągniętych informacji
lista_pacjentow = []
lista_zadan = []

for plik in csv_files:
    # 1. Pobieramy samą nazwę pliku (np. S035_Task1_6.csv)
    nazwa_pliku = os.path.basename(plik)

    # 2. Usuwamy rozszerzenie .csv (zostaje: S035_Task1_6)
    nazwa_bez_rozszerzenia = nazwa_pliku.replace('.csv', '')

    # 3. Rozdzielamy nazwę po znaku podkreślenia
    czesci = nazwa_bez_rozszerzenia.split('_')

    # Zabezpieczenie: sprawdzamy, czy nazwa ma dokładnie 3 części
    if len(czesci) >= 2:
        pacjent = czesci[0]  # np. 'S035'
        zadanie = czesci[1]  # np. 'Task1'

        lista_pacjentow.append(pacjent)
        lista_zadan.append(zadanie)

# 4. Zliczanie wystąpień klas
zliczenia_zadan = Counter(lista_zadan)
unikalni_pacjenci = set(lista_pacjentow)

# 5. Wyświetlanie raportu
print("--- RAPORT KLAS W DATASECIE ---")
print(f"Number of files: {len(csv_files)}")
print(f"Number of patients: {len(unikalni_pacjenci)}\n")

print("Classes (tasks):")
# Wyświetlamy posortowane od najczęstszych
for zadanie, liczba in sorted(zliczenia_zadan.items()):
    print(f" - {zadanie}: {liczba} files")

--- RAPORT KLAS W DATASECIE ---
Number of files: 6583
Number of patients: 109

Classes (tasks):
 - Task1: 822 files
 - Task2: 825 files
 - Task3: 810 files
 - Task4: 819 files
 - Task5: 837 files
 - Task6: 830 files
 - Task7: 825 files
 - Task8: 815 files


In [15]:
test_file = csv_files[289]
df = pd.read_csv(test_file)
df = pd.read_csv(test_file)

print(f"--- RAPORT POPRAWNOŚCI DANYCH ---")
print(f"Badany plik: {os.path.basename(test_file)}\n")

# 1. SPRAWDZENIE KANAŁÓW (Oczekujemy 64 elektrod EEG)
print("1. LICZBA KANAŁÓW:")
# Zazwyczaj jedna lub dwie kolumny to czas i etykieta, reszta to elektrody
liczba_kolumn = len(df.columns)
print(f"- Całkowita liczba kolumn w pliku: {liczba_kolumn}")
print(f"- Pierwsze 3 kolumny: {list(df.columns)[:3]}")
print(f"- Ostatnie 3 kolumny: {list(df.columns)[-3:]}")

# 3. SPRAWDZENIE CZĘSTOTLIWOŚCI PRÓBKOWANIA (Oczekujemy 160 Hz)
print("\n2. CZĘSTOTLIWOŚĆ PRÓBKOWANIA:")
# Szukamy kolumny z czasem, aby sprawdzić odstępy między próbkami
kolumna_czasu = next((col for col in df.columns if col.lower() in ['time', 'czas']), None)

if kolumna_czasu:
    # Obliczamy różnicę czasu między pierwszą a drugą próbką
    roznica_czasu = df[kolumna_czasu].iloc[1] - df[kolumna_czasu].iloc[0]
    obliczone_sfreq = round(1 / roznica_czasu)
    print(f"- Odstęp między próbkami wynosi: {roznica_czasu:.5f} sekundy")
    print(f"- Obliczona częstotliwość próbkowania: {obliczone_sfreq} Hz")
    if obliczone_sfreq == 160:
        print("- [OK] Częstotliwość próbkowania zgadza się z oryginalną specyfikacją PhysioNet.")
    else:
        print("- [UWAGA] Częstotliwość jest inna niż 160 Hz. Być może dane zostały poddane resamplingowi (resampled) przez autora datasetu na Kaggle.")
else:
    print("- Brak kolumny z czasem w pliku. Zakładamy domyślne 160 Hz z opisu datasetu.")
    print(f"- Liczba próbek (wierszy) w pliku: {len(df)}")

--- RAPORT POPRAWNOŚCI DANYCH ---
Badany plik: S034_Task1_1.csv

1. LICZBA KANAŁÓW:
- Całkowita liczba kolumn w pliku: 64
- Pierwsze 3 kolumny: ['Fc5.', 'Fc3.', 'Fc1.']
- Ostatnie 3 kolumny: ['Oz..', 'O2..', 'Iz..']

2. CZĘSTOTLIWOŚĆ PRÓBKOWANIA:
- Brak kolumny z czasem w pliku. Zakładamy domyślne 160 Hz z opisu datasetu.
- Liczba próbek (wierszy) w pliku: 640


## Preprocessing



#### Band-pass filter + features extraction

In [30]:
import os
import numpy as np
import pandas as pd
import mne
import scipy.signal
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler
from mne.decoding import CSP

sfreq = 160
f_low = 8.0
f_high = 45.0

X_features = []
y_labels = []
pominiete_pliki = []

In [31]:
import numpy as np
import scipy.signal
import scipy.stats

def feature_extraction(raw, sfreq):
    data = raw.get_data() # Kształt: (n_channels, n_samples)
    features = []

    freqs, psd = scipy.signal.welch(data, sfreq, nperseg=sfreq)

    bands = {'SMR': (8, 12), 'Low_Beta': (13, 20), 'Gamma': (30, 45)}

    for band_name, (b_low, b_high) in bands.items():
        idx_band = np.logical_and(freqs >= b_low, freqs <= b_high)

        # Średnia moc w podpaśmie
        features.extend(np.mean(psd[:, idx_band], axis=1))

        # Wariancja mocy w podpaśmie
        features.extend(np.var(psd[:, idx_band], axis=1))

    # Skośność
    features.extend(scipy.stats.skew(data, axis=1))

    # Kurtoza
    features.extend(scipy.stats.kurtosis(data, axis=1))

    return np.array(features)

In [32]:
for idx, file_path in enumerate(csv_files):
    file_name = os.path.basename(file_path).replace('.csv', '')
    parts = file_name.split('_')
    if len(parts) < 2:
        continue
    task_label = parts[1]

    try:
        df = pd.read_csv(file_path)

        # Odrzucenie pustych/zbyt krótkich plików
        if df.empty or len(df) < 160:
            pominiete_pliki.append((file_name, "Plik pusty lub za krótki"))
            continue

        ch_names = list(df.columns)
        info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
        data = df.values.T
        raw = mne.io.RawArray(data, info, verbose=False)

        # Filtrowanie SMR, Beta, Gamma (odcina 60Hz z sieci)
        raw.filter(l_freq=f_low, h_freq=f_high, fir_design='firwin', verbose=False)

        # Ekstrakcja cech pojedynczej epoki
        features = feature_extraction(raw, sfreq)

        X_features.append(features)
        y_labels.append(task_label)

    except Exception as e:
        pominiete_pliki.append((file_name, str(e)))
        continue

    if (idx + 1) % 100 == 0:
        print(f"Przetworzono: {idx + 1}/{len(csv_files)} plików...")

Przetworzono: 100/6583 plików...
Przetworzono: 200/6583 plików...
Przetworzono: 300/6583 plików...
Przetworzono: 400/6583 plików...
Przetworzono: 500/6583 plików...
Przetworzono: 600/6583 plików...
Przetworzono: 700/6583 plików...
Przetworzono: 800/6583 plików...
Przetworzono: 900/6583 plików...
Przetworzono: 1000/6583 plików...
Przetworzono: 1100/6583 plików...
Przetworzono: 1200/6583 plików...
Przetworzono: 1300/6583 plików...
Przetworzono: 1400/6583 plików...
Przetworzono: 1500/6583 plików...
Przetworzono: 1600/6583 plików...
Przetworzono: 1700/6583 plików...
Przetworzono: 1800/6583 plików...
Przetworzono: 1900/6583 plików...
Przetworzono: 2000/6583 plików...
Przetworzono: 2100/6583 plików...
Przetworzono: 2200/6583 plików...
Przetworzono: 2300/6583 plików...
Przetworzono: 2400/6583 plików...
Przetworzono: 2500/6583 plików...
Przetworzono: 2600/6583 plików...


/tmp/ipykernel_7005/1742954383.py:22: RuntimeWarning: filter_length (265) is longer than the signal (164), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(l_freq=f_low, h_freq=f_high, fir_design='firwin', verbose=False)


Przetworzono: 2700/6583 plików...
Przetworzono: 2800/6583 plików...
Przetworzono: 2900/6583 plików...
Przetworzono: 3000/6583 plików...
Przetworzono: 3100/6583 plików...
Przetworzono: 3200/6583 plików...
Przetworzono: 3300/6583 plików...
Przetworzono: 3400/6583 plików...
Przetworzono: 3500/6583 plików...


/tmp/ipykernel_7005/1742954383.py:22: RuntimeWarning: filter_length (265) is longer than the signal (164), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(l_freq=f_low, h_freq=f_high, fir_design='firwin', verbose=False)


Przetworzono: 3600/6583 plików...


/tmp/ipykernel_7005/1742954383.py:22: RuntimeWarning: filter_length (265) is longer than the signal (164), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(l_freq=f_low, h_freq=f_high, fir_design='firwin', verbose=False)


Przetworzono: 3700/6583 plików...
Przetworzono: 3800/6583 plików...
Przetworzono: 3900/6583 plików...
Przetworzono: 4000/6583 plików...
Przetworzono: 4100/6583 plików...
Przetworzono: 4200/6583 plików...
Przetworzono: 4300/6583 plików...
Przetworzono: 4400/6583 plików...
Przetworzono: 4500/6583 plików...
Przetworzono: 4600/6583 plików...
Przetworzono: 4700/6583 plików...
Przetworzono: 4800/6583 plików...
Przetworzono: 4900/6583 plików...
Przetworzono: 5000/6583 plików...
Przetworzono: 5100/6583 plików...
Przetworzono: 5200/6583 plików...
Przetworzono: 5300/6583 plików...
Przetworzono: 5400/6583 plików...
Przetworzono: 5500/6583 plików...
Przetworzono: 5600/6583 plików...
Przetworzono: 5700/6583 plików...
Przetworzono: 5800/6583 plików...
Przetworzono: 5900/6583 plików...
Przetworzono: 6000/6583 plików...
Przetworzono: 6100/6583 plików...
Przetworzono: 6200/6583 plików...
Przetworzono: 6300/6583 plików...


/tmp/ipykernel_7005/1742954383.py:22: RuntimeWarning: filter_length (265) is longer than the signal (164), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(l_freq=f_low, h_freq=f_high, fir_design='firwin', verbose=False)


Przetworzono: 6400/6583 plików...
Przetworzono: 6500/6583 plików...


In [33]:
# Raport
print("--- RAPORT Z BŁĘDÓW ---")
if pominiete_pliki:
    print(f"Pominięto {len(pominiete_pliki)} uszkodzonych plików.")
else:
    print("Wszystkie pliki zostały przetworzone bezbłędnie!")

# Synchronizacja listy csv_files
print(f"Liczba plików PRZED: {len(csv_files)}")
uszkodzone_nazwy = [plik[0] for plik in pominiete_pliki]
csv_files = [f for f in csv_files if os.path.basename(f).replace('.csv', '') not in uszkodzone_nazwy]
print(f"Liczba plików PO: {len(csv_files)}")

--- RAPORT Z BŁĘDÓW ---
Pominięto 40 uszkodzonych plików.
Liczba plików PRZED: 6583
Liczba plików PO: 6543


In [34]:
# Konstrukcja macierzy
X = np.array(X_features)
y = np.array(y_labels)

# Standaryzacja cech (wymagane m.in. dla UMAP)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\n--- PREPROCESSING ZAKOŃCZONY ---")
print(f"Kształt macierzy cech X: {X_scaled.shape}")
print(f"Kształt wektora klas y: {y.shape}")


--- PREPROCESSING ZAKOŃCZONY ---
Kształt macierzy cech X: (6543, 512)
Kształt wektora klas y: (6543,)


In [35]:
# last check

# Sprawdzenie wartości pustych (NaN) i nieskończonych (Inf)
if np.isnan(X_scaled).any(): print("BŁĄD: Wykryto wartości NaN w X_scaled!")
if np.isinf(X_scaled).any(): print("BŁĄD: Wykryto wartości Inf w X_scaled!")

# Weryfikacja poprawnego działania StandardScaler
if not np.allclose(np.mean(X_scaled, axis=0), 0, atol=1e-7): print("BŁĄD: Średnia cech nie jest równa 0.")
if not np.allclose(np.std(X_scaled, axis=0), 1, atol=1e-7): print("BŁĄD: Odchylenie standardowe nie jest równe 1.")

# Wykrywanie cech o zerowej wariancji (nie niosących żadnej informacji)
zero_var_idx = np.where(np.var(X, axis=0) == 0)[0]
if len(zero_var_idx) > 0: print(f"BŁĄD: Kolumny o zerowej wariancji do usunięcia (indeksy): {zero_var_idx}")

# Sprawdzenie zbalansowania klas dla modelu LDA
_, counts = np.unique(y, return_counts=True)
if len(set(counts)) > 1: print(f"OSTRZEŻENIE: Klasy są niezbalansowane. Rozkład: {dict(zip(*np.unique(y, return_counts=True)))}")

OSTRZEŻENIE: Klasy są niezbalansowane. Rozkład: {np.str_('Task1'): np.int64(818), np.str_('Task2'): np.int64(819), np.str_('Task3'): np.int64(805), np.str_('Task4'): np.int64(814), np.str_('Task5'): np.int64(832), np.str_('Task6'): np.int64(825), np.str_('Task7'): np.int64(819), np.str_('Task8'): np.int64(811)}


In [36]:
# save preprocessed data to file

liczba_cech = X_scaled.shape[1]
nazwy_kolumn = [f"Cecha_{i+1}" for i in range(liczba_cech)]

df_gotowe = pd.DataFrame(X_scaled, columns=nazwy_kolumn)

df_gotowe['Label'] = y

sciezka_csv = '/content/drive/MyDrive/dataset_eeg_features.csv'
df_gotowe.to_csv(sciezka_csv, index=False)

print(f"Dane gotowe na GitHuba zapisane w: {sciezka_csv}")

Dane gotowe na GitHuba zapisane w: /content/drive/MyDrive/dataset_eeg_features.csv


In [38]:
print(df_gotowe.head(20))

     Cecha_1   Cecha_2   Cecha_3   Cecha_4   Cecha_5   Cecha_6   Cecha_7  \
0  -0.660561 -0.687001 -0.552521 -0.582926 -0.546565 -0.490115 -0.641453   
1  -0.576983 -0.605609 -0.592273 -0.583469 -0.543868 -0.366741 -0.494908   
2  -0.125985 -0.013569  0.034881  0.142811  0.199140  0.042785 -0.222047   
3  -0.180234 -0.340591 -0.262764 -0.287211 -0.263938 -0.213571 -0.210655   
4  -0.779998 -0.803818 -0.787602 -0.779821 -0.779796 -0.549005 -0.761815   
5   0.077979 -0.204327 -0.392519 -0.342075 -0.411178 -0.205281 -0.297259   
6  -0.464949 -0.470609 -0.423064 -0.350675 -0.381136 -0.355462 -0.426279   
7  -0.550746 -0.632122 -0.609587 -0.626607 -0.604371 -0.423515 -0.609094   
8  -0.776969 -0.791067 -0.789007 -0.772420 -0.772937 -0.550071 -0.764980   
9  -0.595579 -0.621459 -0.651140 -0.657975 -0.646582 -0.468111 -0.623302   
10 -0.810753  0.151148  1.049702 -0.374794 -0.333562  0.344159 -0.226842   
11 -0.788550 -0.807447 -0.804710 -0.786618 -0.786351 -0.563501 -0.804765   
12 -0.535758